In [299]:
import pandas as pd
import numpy as np

from sklearn.metrics import accuracy_score, log_loss
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import xgboost as xgb

In [311]:
path = r"data\dataset.csv"
with open(path) as file:
    df = pd.read_csv(file)
df.head()

,year,month,day,home_team,away_team,stage,result,shootouts,home_score,home_is_host,...,away_draws_last_4y,away_wc_titles_before,away_total_market_value_eur,away_avg_age,away_wc_participations_before,away_groups_passed_before,away_round16_before,away_quarterfinals_before,away_semifinals_before,away_finals_before
0,2006,6,9,Germany,Costa Rica,0,0.0,0,4.0,1,...,11,0,18400000.0,24.7,2,1,1,0,0,0
1,2006,6,9,Poland,Ecuador,0,1.0,0,0.0,0,...,12,0,387830000.0,25.7,1,0,0,0,0,0
2,2006,6,10,Argentina,Ivory Coast,0,0.0,0,2.0,0,...,8,0,410900000.0,25.8,0,0,0,0,0,0
3,2006,6,10,England,Paraguay,0,0.0,0,1.0,0,...,17,0,141850000.0,29.1,6,3,3,0,0,0
4,2006,6,10,Trinidad and Tobago,Sweden,0,2.0,0,0.0,0,...,18,0,363780000.0,26.3,10,7,3,4,3,1


In [312]:
# selecting features and target
features = df[['year', 'shootouts', 'home_is_host', 'home_goals_scored_last_4y',
       'home_goals_received_last_4y', 'home_wins_last_4y',
       'home_losses_last_4y', 'home_draws_last_4y', 'home_wc_titles_before',
       'home_total_market_value_eur', 'home_avg_age',
       'home_wc_participations_before', 'home_groups_passed_before',
       'home_round16_before', 'home_quarterfinals_before',
       'home_semifinals_before', 'home_finals_before',
       'away_is_host', 'away_goals_scored_last_4y',
       'away_goals_received_last_4y', 'away_wins_last_4y',
       'away_losses_last_4y', 'away_draws_last_4y', 'away_wc_titles_before',
       'away_total_market_value_eur', 'away_avg_age',
       'away_wc_participations_before', 'away_groups_passed_before',
       'away_round16_before', 'away_quarterfinals_before',
       'away_semifinals_before', 'away_finals_before']]
target = df[['year', 'result']]

In [313]:
# standardizing features
features.columns

Index(['year', 'shootouts', 'home_is_host', 'home_goals_scored_last_4y',
       'home_goals_received_last_4y', 'home_wins_last_4y',
       'home_losses_last_4y', 'home_draws_last_4y', 'home_wc_titles_before',
       'home_total_market_value_eur', 'home_avg_age',
       'home_wc_participations_before', 'home_groups_passed_before',
       'home_round16_before', 'home_quarterfinals_before',
       'home_semifinals_before', 'home_finals_before', 'away_is_host',
       'away_goals_scored_last_4y', 'away_goals_received_last_4y',
       'away_wins_last_4y', 'away_losses_last_4y', 'away_draws_last_4y',
       'away_wc_titles_before', 'away_total_market_value_eur', 'away_avg_age',
       'away_wc_participations_before', 'away_groups_passed_before',
       'away_round16_before', 'away_quarterfinals_before',
       'away_semifinals_before', 'away_finals_before'],
      dtype='object')

In [314]:
scaler = StandardScaler()
to_be_standardized = ["home_goals_scored_last_4y", "home_goals_received_last_4y", "home_wins_last_4y", "home_losses_last_4y",
                        "home_draws_last_4y", "home_wc_titles_before", "home_total_market_value_eur", "home_avg_age", 
                        "home_wc_participations_before", "home_groups_passed_before", "home_round16_before", "home_quarterfinals_before",
                        "home_semifinals_before", "home_finals_before", 'away_goals_scored_last_4y', 'away_goals_received_last_4y',
                        'away_wins_last_4y', 'away_losses_last_4y', 'away_draws_last_4y',
                        'away_wc_titles_before', 'away_total_market_value_eur', 'away_avg_age',
                        'away_wc_participations_before', 'away_groups_passed_before',
                        'away_round16_before', 'away_quarterfinals_before',
                        'away_semifinals_before', 'away_finals_before']


In [315]:
# Using Logistic regression 
# Accuracy and loss are computed on the train set

# to be modified: now it outputs a binary variable (it is totally wrong), I should implement a softmax regression with 3 classes
# try a ridge regularization and see what happens

X_train = features[(features['year'] != 2026)].drop(columns=['year'])
Y_train = target.loc[target['year'] != 2026, 'result']

X_test = features[(features['year'] == 2026)].drop(columns=['year'])
Y_test = target.loc[target['year'] == 2026, 'result']

# standardizing features
X_train_scaled = X_train.copy()
X_train_scaled[to_be_standardized] = X_train_scaled[to_be_standardized].astype(float)
X_train_scaled.loc[:, to_be_standardized] = scaler.fit_transform(X_train_scaled[to_be_standardized])

X_test_scaled = X_test.copy()
X_test_scaled[to_be_standardized] = X_test_scaled[to_be_standardized].astype(float)
X_test_scaled.loc[:, to_be_standardized] = scaler.transform(X_test_scaled[to_be_standardized])

model = LogisticRegression(max_iter=100)
_ = model.fit(X_train_scaled, Y_train)

prob = _.predict_proba(X_train_scaled)
preds = _.predict(X_train_scaled)

acc = accuracy_score(Y_train, preds)
loss = log_loss(Y_train, prob)

print(f"World Cup 2006-2022: Accuracy: {acc:.4f}, LogLoss: {loss:.4f}")

World Cup 2006-2022: Accuracy: 0.6281, LogLoss: 0.8364


In [316]:
# computing accuracy and loss on test set

X_train = features[(features['year'] != 2026)].drop(columns=['year'])
Y_train = target[(target['year'] != 2026)]['result']

X_test = features[(features['year'] == 2026)].drop(columns=['year'])
Y_test = target[(target['year'] == 2026)]['result']

# standardizing features
X_train_scaled = X_train.copy()
X_train_scaled[to_be_standardized] = X_train_scaled[to_be_standardized].astype(float)
X_train_scaled.loc[:, to_be_standardized] = scaler.fit_transform(X_train_scaled[to_be_standardized])

X_test_scaled = X_test.copy()
X_test_scaled[to_be_standardized] = X_test_scaled[to_be_standardized].astype(float)
X_test_scaled.loc[:, to_be_standardized] = scaler.transform(X_test_scaled[to_be_standardized])

model = LogisticRegression(max_iter=100)
_ = model.fit(X_train_scaled, Y_train)

prob = _.predict_proba(X_test_scaled)
preds = _.predict(X_test_scaled)

acc = accuracy_score(Y_test, preds)
loss = log_loss(Y_test, prob)

print(f"World Cup 2026: Accuracy: {acc:.4f}, LogLoss: {loss:.4f}")

World Cup 2026: Accuracy: 0.6600, LogLoss: 0.8639


In [317]:
# Trying XGBoost
# Accuracy and loss are computed on the training set

X_train = features[(features['year'] != 2026)].drop(columns=['year'])
Y_train = target[(target['year'] != 2026)]['result']

X_test = features[(features['year'] == 2026)].drop(columns=['year'])
Y_test = target[(target['year'] == 2026)]['result']

# standardizing features
X_train_scaled = X_train.copy()
X_train_scaled[to_be_standardized] = X_train_scaled[to_be_standardized].astype(float)
X_train_scaled.loc[:, to_be_standardized] = scaler.fit_transform(X_train_scaled[to_be_standardized])

X_test_scaled = X_test.copy()
X_test_scaled[to_be_standardized] = X_test_scaled[to_be_standardized].astype(float)
X_test_scaled.loc[:, to_be_standardized] = scaler.transform(X_test_scaled[to_be_standardized])

# fitting model and computing probabilities
model = xgb.XGBClassifier(
        objective="multi:softprob", num_class=3,
        max_depth=3, min_child_weight=10,
        subsample=0.7, colsample_bytree=0.7,
        reg_lambda=30, n_estimators=10,
        eval_metric="mlogloss", random_state=42,
)
model.fit(X_train_scaled, Y_train)

prob = model.predict_proba(X_train_scaled)

# creating a dataframe to store the probabilities
df_prob = pd.DataFrame(prob, columns = ["home_prob", "away_prob", "draw_prob"])
df_prob["stage"] = df[df["year"] < 2026]["stage"].values

# compute normalization only for knockouts
sum_no_draw = df_prob["home_prob"] + df_prob["away_prob"]
df_prob["home_ko_prob"] = df_prob["home_prob"] / sum_no_draw
df_prob["away_ko_prob"] = df_prob["away_prob"] / sum_no_draw

def final_output(row):
    if row["stage"] == 1:
        return [row["home_ko_prob"], row["away_ko_prob"]]
    else:
        return [row["home_prob"], row["draw_prob"], row["away_prob"]]

df_prob["probabilities"] = df_prob.apply(final_output, axis=1)

# inferring prediction from probabilities
def predict(row):
    if row["stage"] == 1:
        if row["home_ko_prob"] > row["away_ko_prob"]:
            return 0 # home_team wins
        else:
            return 1 # away_team wins
    else:
        classes = [0, 1, 2] # for groups
        groups_prob = [row["home_prob"], row["away_prob"], row["draw_prob"]]
        return classes[np.argmax(groups_prob)]

df_prob["prediction"] = df_prob.apply(predict, axis=1)
preds = df_prob["prediction"]

acc = accuracy_score(Y_train, preds)

# computing loss for groups 
is_group = df_prob["stage"] == 0
target_groups = Y_train[is_group]
probs_groups = df_prob[is_group][["home_prob", "away_prob", "draw_prob"]].values
loss_groups = log_loss(target_groups, probs_groups)

# computing loss for knockouts
is_ko = df_prob["stage"] == 1
target_ko = Y_train[is_ko]
probs_ko = df_prob[is_ko][["home_ko_prob", "away_ko_prob"]].values
loss_ko = log_loss(target_ko, probs_ko)

print(f"World Cup 2006-2022: Accuracy: {acc:.4f} | LogLoss groups: {loss_groups:.4f} | LogLoss ko: {loss_ko:.4f}")

World Cup 2006-2022: Accuracy: 0.6500 | LogLoss groups: 0.9169 | LogLoss ko: 0.5088


In [ ]:
# Computing accuracy and loss on the test set

X_train = features[(features['year'] != 2026)].drop(columns=['year'])
Y_train = target[(target['year'] != 2026)]['result']

X_test = features[(features['year'] == 2026)].drop(columns=['year'])
Y_test = target[(target['year'] == 2026)]['result'].reset_index(drop=True) 
# reset_index to align the index with is_group to correctly index it when computing losses

# standardizing features
X_train_scaled = X_train.copy()
X_train_scaled[to_be_standardized] = X_train_scaled[to_be_standardized].astype(float)
X_train_scaled.loc[:, to_be_standardized] = scaler.fit_transform(X_train_scaled[to_be_standardized])

X_test_scaled = X_test.copy()
X_test_scaled[to_be_standardized] = X_test_scaled[to_be_standardized].astype(float)
X_test_scaled.loc[:, to_be_standardized] = scaler.transform(X_test_scaled[to_be_standardized])

# fitting model and computing probabilities
model = xgb.XGBClassifier(
        objective="multi:softprob", num_class=3,
        max_depth=3, min_child_weight=10,
        subsample=0.7, colsample_bytree=0.7,
        reg_lambda=30.0, n_estimators=10,
        eval_metric="mlogloss", random_state=42,
)
model.fit(X_train_scaled, Y_train)

prob = model.predict_proba(X_test_scaled)

# creating a dataframe to store the probabilities
df_prob = pd.DataFrame(prob, columns = ["home_prob", "away_prob", "draw_prob"])
df_prob["stage"] = df[df["year"] == 2026]["stage"].values

# compute normalization only for knockouts
sum_no_draw = df_prob["home_prob"] + df_prob["away_prob"]
df_prob["home_ko_prob"] = df_prob["home_prob"] / sum_no_draw
df_prob["away_ko_prob"] = df_prob["away_prob"] / sum_no_draw

def final_output(row):
    if row["stage"] == 1:
        return [row["home_ko_prob"], row["away_ko_prob"]]
    else:
        return [row["home_prob"], row["away_prob"], row["draw_prob"]]

df_prob["probabilities"] = df_prob.apply(final_output, axis=1)

# inferring prediction from probabilities
def predict(row):
    if row["stage"] == 1:
        if row["home_ko_prob"] > row["away_ko_prob"]:
            return 0 # home_team wins
        else:
            return 1 # away_team wins
    else:
        classes = [0, 1, 2] # for groups
        groups_prob = [row["home_prob"], row["away_prob"], row["draw_prob"]]
        return classes[np.argmax(groups_prob)]
    
df_prob["prediction"] = df_prob.apply(predict, axis=1)
preds = df_prob["prediction"]

acc = accuracy_score(Y_test, preds)

# computing loss for groups 
is_group = df_prob["stage"] == 0
target_groups = Y_test[is_group]
probs_groups = df_prob[is_group][["home_prob", "away_prob", "draw_prob"]].values
loss_groups = log_loss(target_groups, probs_groups)

# computing loss for knockouts
is_ko = df_prob["stage"] == 1
target_ko = Y_test[is_ko]
probs_ko = df_prob[is_ko][["home_ko_prob", "away_ko_prob"]].values
loss_ko = log_loss(target_ko, probs_ko)

print(f"World Cup 2026: Accuracy: {acc:.4f} | LogLoss groups: {loss_groups:.4f} | LogLoss ko: {loss_ko:.4f}")

World Cup 2026: Accuracy: 0.6600 | LogLoss groups: 0.9699 | LogLoss ko: 0.6060


In [ ]:
# Predicting only knockouts
# selecting features and target
features = df[['year', 'shootouts', 'home_is_host', 'home_goals_scored_last_4y',
       'home_goals_received_last_4y', 'home_wins_last_4y',
       'home_losses_last_4y', 'home_draws_last_4y', 'home_wc_titles_before',
       'home_total_market_value_eur', 'home_avg_age',
       'home_wc_participations_before', 'home_groups_passed_before',
       'home_round16_before', 'home_quarterfinals_before',
       'home_semifinals_before', 'home_finals_before',
       'away_is_host', 'away_goals_scored_last_4y',
       'away_goals_received_last_4y', 'away_wins_last_4y',
       'away_losses_last_4y', 'away_draws_last_4y', 'away_wc_titles_before',
       'away_total_market_value_eur', 'away_avg_age',
       'away_wc_participations_before', 'away_groups_passed_before',
       'away_round16_before', 'away_quarterfinals_before',
       'away_semifinals_before', 'away_finals_before', 'stage']]
target = df[['year', 'result', 'stage']]

In [ ]:
# Computing accuracy and loss on the test set

X_train = features[(features['year'] != 2026)].drop(columns=['year', 'stage'])
Y_train = target[(target['year'] != 2026)]['result']

X_test = features[(features['year'] == 2026)].drop(columns=['year'])
X_test = X_test[X_test["stage"] == 1].drop(columns = "stage")
Y_test = target[(target['year'] == 2026)][['result', 'stage']].reset_index(drop=True) 
# reset_index to align the index with is_group to correctly index it when computing losses
Y_test = Y_test[Y_test["stage"] == 1].drop(columns = "stage")

# # standardizing features
# X_train_scaled = X_train.copy()
# X_train_scaled[to_be_standardized] = X_train_scaled[to_be_standardized].astype(float)
# X_train_scaled.loc[:, to_be_standardized] = scaler.fit_transform(X_train_scaled[to_be_standardized])

# X_test_scaled = X_test.copy()
# X_test_scaled[to_be_standardized] = X_test_scaled[to_be_standardized].astype(float)
# X_test_scaled.loc[:, to_be_standardized] = scaler.transform(X_test_scaled[to_be_standardized])

# if you want to scale remember to change variables' names in the model

# fitting model and computing probabilities
model = xgb.XGBClassifier(
        objective="multi:softprob", 
        max_depth=3, min_child_weight=10,
        subsample=0.7, colsample_bytree=0.7,
        reg_lambda=20.0, n_estimators=10,
        eval_metric="mlogloss", random_state=42,
)
model.fit(X_train, Y_train)

prob = model.predict_proba(X_test)

# creating a dataframe to store the probabilities
df_prob = pd.DataFrame(prob, columns = ["home_prob", "away_prob", "draw_prob"])
df_prob["stage"] = df[(df["year"] == 2026) & (df["stage"] == 1)]["stage"].values

# compute normalization for knockouts
sum_no_draw = df_prob["home_prob"] + df_prob["away_prob"]
df_prob["home_ko_prob"] = df_prob["home_prob"] / sum_no_draw
df_prob["away_ko_prob"] = df_prob["away_prob"] / sum_no_draw

def final_output(row):
    return [row["home_ko_prob"], row["away_ko_prob"]]

df_prob["probabilities"] = df_prob.apply(final_output, axis=1)

# inferring prediction from probabilities
def predict(row):
    if row["home_ko_prob"] > row["away_ko_prob"]:
        return 0 # home_team wins
    else:
        return 1 # away_team wins
    
df_prob["prediction"] = df_prob.apply(predict, axis=1)
preds = df_prob["prediction"]

acc = accuracy_score(Y_test, preds)

# computing loss for knockouts
probs_ko = df_prob[["home_ko_prob", "away_ko_prob"]].values
loss_ko = log_loss(Y_test, probs_ko)

print(f"World Cup 2026: Accuracy: {acc:.4f} | LogLoss: {loss_ko:.4f}")

World Cup 2026: Accuracy: 0.6786 | LogLoss: 0.6075


In [310]:
# To be done: implementing a random forest